# INSIDER ALPHA — Real Multi-Agent System

## Single Market Deep Dive

This notebook demonstrates the **real agents** from the polymarket code repository analyzing ONE prediction market.

**What you'll see:**
- Agent A: Real insider risk analysis (LLM call)
- Agent B: Real market behavior analysis (LLM + price tools)
- Revision Agent: Real pattern detection & QA
- Decision Agent: Real weighted investment decision

**Duration:** ~3 minutes (1 LLM call per agent × 4 = 4 API calls)

---

## 1. Setup: Paths and Imports

In [54]:
import sys
from pathlib import Path
import os
from dotenv import load_dotenv
from datetime import datetime, timezone
import json

# Setup paths — works whether the notebook is run from:
#   (a) the project root  (polymarket-events/)
#   (b) a sibling demo/   folder

current_file = Path.cwd()

# Locate demo_dir (where data/ lives)
if (current_file / "demo" / "data").exists():
    # Running from project root — demo data is in ./demo/data/
    demo_dir = current_file / "demo"
elif (current_file / "data" / "exports").exists():
    # Running from inside the demo/ folder directly
    demo_dir = current_file
else:
    demo_dir = current_file / "demo"  # fallback

# Project dir = parent of demo_dir
project_dir = demo_dir.parent

# Locate the code folder (has src/ai_layer/) — check project_dir itself first,
# then its sub-directories (handles both single-repo and nested structures)
candidate_dirs = []

# Case A: notebook is at project root → src/ lives directly in project_dir
if (project_dir / "src" / "ai_layer").exists():
    candidate_dirs.append(project_dir)

# Case B: project_dir has child repos (classic demo-alongside-repo layout)
for item in project_dir.iterdir():
    if item.is_dir() and (item / "src" / "ai_layer").exists():
        if item not in candidate_dirs:
            candidate_dirs.append(item)

if not candidate_dirs:
    raise FileNotFoundError(
        f"\n❌ ERROR: Cannot find the polymarket code folder!\n"
        f"Expected a folder with \'src/ai_layer/\' inside: {project_dir}\n"
        f"Make sure you have cloned the polymarket repository.\n"
    )

if len(candidate_dirs) > 1:
    remove_pandas = [d for d in candidate_dirs if "remove-pandas" in d.name]
    master_dir = remove_pandas[0] if remove_pandas else max(candidate_dirs, key=lambda p: p.stat().st_mtime)
else:
    master_dir = candidate_dirs[0]

print(f"📁 Demo dir:      {demo_dir}")
print(f"📁 Project dir:   {project_dir}")
print(f"📁 Master dir:    {master_dir}")
print(f"\n✅ Demo data found: {(demo_dir / \'data/exports/polymarket_tagged_sample.parquet\').exists()}")
print(f"✅ Master dir exists: {master_dir.exists()}")

# Add to Python path
sys.path.insert(0, str(master_dir))
sys.path.insert(0, str(project_dir))

# Load .env — check demo_dir first, then project root
env_path = demo_dir / ".env"
if not env_path.exists():
    env_path = project_dir / ".env"
load_dotenv(env_path)
print(f"\n✅ .env loaded from: {env_path}")
print(f"✅ OPENAI_API_KEY configured: {bool(os.getenv(\'OPENAI_API_KEY\'))}")

if not os.getenv(\'OPENAI_API_KEY\'):
    raise ValueError(
        f"\n❌ ERROR: OPENAI_API_KEY not found!\n"
        f"Create a .env file at {project_dir / \'.env\'} with:\n"
        f"  OPENAI_API_KEY=sk-proj-YOUR_KEY_HERE\n"
    )


📁 Demo dir:      c:\Users\jsilvaec\OneDrive - The University of Chicago\Desktop\Personal Projects\Polymarket Events
📁 Project dir:   c:\Users\jsilvaec\OneDrive - The University of Chicago\Desktop\Personal Projects
📁 Master dir:    c:\Users\jsilvaec\OneDrive - The University of Chicago\Desktop\Personal Projects\Polymarket Events

✅ Demo dir exists:    True
✅ Master dir exists:  True
✅ Found code folder:  Polymarket Events

✅ .env loaded from: c:\Users\jsilvaec\OneDrive - The University of Chicago\Desktop\Personal Projects\Polymarket Events\.env
✅ OPENAI_API_KEY configured: True


---

## 2. Import Real Agents from Master 2

In [55]:
# Import real agents
from src.ai_layer.agent_a.agent import agent_a_initial
from src.ai_layer.agent_a.schemas import AgentAInputPackage
from src.ai_layer.agent_a.params import AgentAParams

from src.ai_layer.agent_b.agent import agent_b_initial
from src.ai_layer.agent_b.schemas import AgentBInputPackage
from src.ai_layer.agent_b.params import AgentBParams

from src.ai_layer.revision_agent import revision_agent_deterministic

from src.ai_layer.decision_agent.agent import decision_agent_deterministic
from src.ai_layer.decision_agent.schemas import DecisionAgentInputPackage
from src.ai_layer.decision_agent.params import DecisionAgentParams

print("✅ All agents imported successfully")

✅ All agents imported successfully


---

## 3. Load Real Market Data

In [62]:
import pandas as pd
import sqlite3
from datetime import datetime, timezone, timedelta

# Load real polymarket data
parquet_path = demo_dir / "data/exports/polymarket_tagged_sample.parquet"
df_raw = pd.read_parquet(parquet_path)

print(f"📊 Loaded {len(df_raw):,} markets from parquet")
print(f"Columns: {list(df_raw.columns)[:8]}...\n")

# Load price history database
db_path = demo_dir / "data/price_history.db"
conn = sqlite3.connect(db_path)
price_df = pd.read_sql_query("SELECT market_id, timestamp, price FROM price_history", conn)
conn.close()

print(f"📊 Loaded {len(price_df):,} price points from price_history.db\n")

# Filter markets
df_filtered = df_raw[
    (df_raw['volume'] >= 20_000) &
    (df_raw['status'] == 'closed') &
    (df_raw['end_date'].notna())
].copy()

# Remove boring categories
boring = ['Crypto', 'Elections', 'Sports', 'NBA', 'NFL']
for cat in boring:
    df_filtered = df_filtered[~df_filtered['category'].str.contains(cat, case=False, na=False)]

# Sort by volume (descending) to get highest-volume markets first
df_filtered = df_filtered.sort_values('volume', ascending=False).reset_index(drop=True)

market_id = "523151"
market = df_filtered[df_filtered['market_id'] == market_id].iloc[0]

print("-" * 100)
print(f"\n🎯 ANALYZING MARKET:")
print(f"\nMarket ID: {market['market_id']}")
print(f"Question: {market['question']}")
print(f"Category: {market['category']}")
print(f"Volume: ${market['volume']:,.0f}")
print(f"Status: {market['status']}")
print(f"End Date: {market['end_date']}")

# Get price history for this market
market_id_str = str(int(market['market_id']))
market_prices = price_df[price_df['market_id'] == market_id_str].copy()
print(f"\n💰 Price history points available: {len(market_prices)}")
if len(market_prices) > 0:
    print(f"   Price range: ${market_prices['price'].min():.3f} - ${market_prices['price'].max():.3f}")

📊 Loaded 15,303 markets from parquet
Columns: ['platform', 'market_id', 'ticker', 'event_id', 'event_title', 'question', 'slug', 'category']...

📊 Loaded 689,530 price points from price_history.db

----------------------------------------------------------------------------------------------------

🎯 ANALYZING MARKET:

Market ID: 523151
Question: Will GPT-4.5 be released by February 28?
Category: 
Volume: $141,504
Status: closed
End Date: 2025-02-28 12:00:00+00:00

💰 Price history points available: 32
   Price range: $0.105 - $0.996


---

## 4. AGENT A: Insider Risk Assessment

Real LLM call to analyze insider risk from market description.

In [68]:
print("🔍 AGENT A: Insider Risk Assessment\n")
print(f"Input: Market title + description (blind to price/volume)\n")

# Create Agent A input
pkg_a = AgentAInputPackage(
    market_id=str(market['market_id']),
    question=market['question'],
    description=market['question'],  # description not always available
    category=market.get('category', ''),
    tags=market.get('tags', '').split('|')[:5] if isinstance(market.get('tags'), str) else [],
    platform="polymarket",
    end_date=pd.Timestamp(market['end_date']).to_pydatetime().replace(tzinfo=timezone.utc) if pd.notna(market['end_date']) else datetime.now(timezone.utc),
)

# Run Agent A (REAL LLM call)
report_a = agent_a_initial(pkg_a, AgentAParams())

print(f"✅ Agent A Complete\n")
print(f"Insider Risk Score: {report_a.insider_risk_score}/10")
print(f"Confidence: {report_a.confidence}")
print(f"\nReasoning:")
print(f"  {report_a.reasoning[:300]}...\n")
print(f"Info Holders: {', '.join(report_a.info_holders[:3])}")
print(f"Leak Vectors: {', '.join(report_a.leak_vectors[:2])}")

🔍 AGENT A: Insider Risk Assessment

Input: Market title + description (blind to price/volume)

✅ Agent A Complete

Insider Risk Score: 6/10
Confidence: medium

Reasoning:
  The release of GPT-4.5 is likely known to a moderate-size group, including OpenAI executives and developers, who may have advance knowledge of the timeline. While there is a financial incentive for insiders to act on this information, the presence of legal and reputational barriers may reduce the li...

Info Holders: OpenAI executives, product managers, developers
Leak Vectors: social media posts by insiders, discussions in industry conferences


---

## 5. AGENT B: Market Behavior Analysis

Real analysis of price/volume anomalies (real price tools).

In [69]:
print("🔍 AGENT B: Market Behavior Analysis\n")
print(f"Input: Price history (last 5 days) + volume data\n")

# Get price history for this market and convert to PricePoint objects
from src.data_layer.models import PricePoint

# Build price history list
price_list = []
if len(market_prices) > 0:
    market_prices_sorted = market_prices.sort_values('timestamp')
    for _, row in market_prices_sorted.iterrows():
        # Convert Unix timestamp to datetime
        ts = datetime.fromtimestamp(row['timestamp'], tz=timezone.utc)
        price = float(row['price'])
        price_list.append(PricePoint(timestamp=ts, price=price, raw_price=price))

# Match backtest: evaluation_date = end_date - 24h, window = [end_date-120h, end_date-24h]
end_dt = pd.Timestamp(market['end_date']).to_pydatetime().replace(tzinfo=timezone.utc) if pd.notna(market['end_date']) else datetime.now(timezone.utc)
eval_date = end_dt - timedelta(hours=24)
window_start = end_dt - timedelta(hours=120)

# Filter price history to backtest window
price_list_window = [p for p in price_list if window_start <= p.timestamp <= eval_date]

if len(price_list_window) > 0:
    print(f"✅ Using {len(price_list_window)} price points in [end-120h, end-24h] window")
    first_ts = price_list_window[0].timestamp
    last_ts = price_list_window[-1].timestamp
    print(f"   Time range: {first_ts.date()} to {last_ts.date()}")
    print(f"   Price range: ${min(p.price for p in price_list_window):.3f} - ${max(p.price for p in price_list_window):.3f}")
else:
    print(f"⚠️  No price data in backtest window, using all available ({len(price_list)} points)")
    price_list_window = price_list

# Current price = price at evaluation_date (last point in window), matching backtest
current_price = price_list_window[-1].price if price_list_window else (float(market['yes_price']) if pd.notna(market['yes_price']) else 0.5)

# Volume and market age, matching backtest
volume_raw = market.get('volume', None)
volume_total = float(volume_raw) if pd.notna(volume_raw) else None

market_age_days = None
if pd.notna(market.get('start_date')) and pd.notna(market.get('end_date')):
    start_dt = pd.Timestamp(market['start_date']).to_pydatetime().replace(tzinfo=timezone.utc)
    market_age_days = (end_dt - start_dt).total_seconds() / 86_400

# Create Agent B input matching backtest exactly
pkg_b = AgentBInputPackage(
    evaluation_date=eval_date,
    end_date=end_dt,
    price_history=price_list_window,
    current_price=current_price,
    volume_total_usd=volume_total,
    market_age_days=market_age_days,
)

# Run Agent B (REAL LLM call with backtest-matching inputs)
try:
    report_b = agent_b_initial(pkg_b, AgentBParams())
    print(f"\n✅ Agent B Complete\n")
    print(f"Behavior Score: {report_b.behavior_score}/10")
    print(f"Signal Direction: {report_b.signal_direction}")
    print(f"Confidence: {report_b.confidence}")
    print(f"\nReasoning: {report_b.reasoning[:250]}...")
    print(f"\nKey Findings:")
    for finding in report_b.key_findings[:3]:
        print(f"  • {finding}")
except Exception as e:
    print(f"⚠️  Agent B error: {str(e)[:200]}")

🔍 AGENT B: Market Behavior Analysis

Input: Price history (last 5 days) + volume data

✅ Using 8 price points in [end-120h, end-24h] window
   Time range: 2025-02-23 to 2025-02-27
   Price range: $0.125 - $0.825

✅ Agent B Complete

Behavior Score: 9/10
Signal Direction: YES
Confidence: high

Reasoning: The price jump of 70pp is substantial and sustained, indicating a strong upward movement. The momentum analysis aligns with this price movement, showing a consistent upward trend. Given the proximity to market close and the absence of contradictory s...

Key Findings:
  • Detected a significant price jump of 70pp from 0.125 to 0.825, sustained and within 24 hours of market close.
  • Momentum analysis shows a consistent upward trend with a strong slope, supporting the price jump.


C:\Users\jsilvaec\AppData\Roaming\Python\Python314\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=_LLMAgentBResponse(signal...onverging towards 1.0.'), input_type=_LLMAgentBResponse])
  return self.__pydantic_serializer__.to_python(


---

## 6. REVISION AGENT: Pattern Detection & QA

Real LLM-powered validation of coherence between Agents A & B.

In [70]:
print("🔍 REVISION AGENT: Pattern Detection & QA\n")
print(f"Input: Agent A + Agent B reports\n")

try:
    # Run Revision Agent (deterministic — same as backtest, no LLM needed)
    revision_out = revision_agent_deterministic(
        agent_a_report=report_a.model_dump(),
        agent_b_report=report_b.model_dump() if 'report_b' in locals() else {},
    )
    
    print(f"✅ Revision Agent Complete\n")
    print(f"Detected Pattern: {revision_out.revision_flag}")
    print(f"\nExplanation: {revision_out.flag_explanation[:200]}...")
    print(f"\nRevision Notes:")
    for line in revision_out.revision_notes.split("\n")[:4]:
        if line.strip():
            print(f"  {line}")
    print(f"\nRecommendation to Decision Agent: {revision_out.recommendation_to_decision_agent}")
except Exception as e:
    print(f"⚠️  Revision Agent error: {str(e)[:150]}")

🔍 REVISION AGENT: Pattern Detection & QA

Input: Agent A + Agent B reports

✅ Revision Agent Complete

Detected Pattern: NONE

Explanation: No cross-pattern detected (A=6, B=9, dir=YES)....

Revision Notes:
  No cross-pattern detected (A=6, B=9, dir=YES).

Recommendation to Decision Agent: GO_EVALUATE


---

## 7. DECISION AGENT: Final Investment Decision

Real Bayesian-weighted decision: GO or SKIP?

In [71]:
print("🎯 DECISION AGENT: Final Investment Decision\n")
print(f"Input: Agent A + Agent B + Revision reports\n")

try:
    now = eval_date  # Use backtest-matching evaluation date, not datetime.now()
    
    pkg_dec = DecisionAgentInputPackage(
        revision_flag=revision_out.revision_flag if 'revision_out' in locals() else "NONE",
        flag_explanation=revision_out.flag_explanation if 'revision_out' in locals() else "",
        agent_a_report=report_a.model_dump(),
        agent_b_report=report_b.model_dump() if 'report_b' in locals() else {},
        revision_notes=revision_out.revision_notes if 'revision_out' in locals() else "",
        recommendation_to_decision_agent=revision_out.recommendation_to_decision_agent if 'revision_out' in locals() else "GO_EVALUATE",
        current_market_price=current_price,
        evaluation_date=eval_date,
        end_date=end_dt,
        market_id=str(market['market_id']),
    )
    
    # Use aggressive config — matches the backtest config that produced GO signals
    aggressive_params = DecisionAgentParams(
        min_a_score=3, min_b_score=3, go_score_threshold=5.0, min_edge_pp=3.0
    )
    decision = decision_agent_deterministic(pkg_dec, aggressive_params)
    
    print(f"✅ Decision Agent Complete\n")
    decision_symbol = "✅ GO" if decision.decision == "GO" else "❌ SKIP"
    print(f"DECISION: {decision_symbol}")
    print(f"\nAnalysis:")
    if decision.analysis:
        print(f"  Weighted Score: {decision.analysis.get('weighted_score', '?')}/10")
    print(f"  Bet Direction: {decision.bet_direction}")
    print(f"  Entry Price: ${current_price:.2f}")
    
    if decision.recommendation:
        print(f"\nRecommendation:")
        print(f"  Action: {decision.recommendation.get('action', '?')}")
        print(f"  Risk Grade: {decision.recommendation.get('risk_grade', '?')}")
        
except Exception as e:
    print(f"⚠️  Decision Agent error: {str(e)[:150]}")

🎯 DECISION AGENT: Final Investment Decision

Input: Agent A + Agent B + Revision reports

✅ Decision Agent Complete

DECISION: ✅ GO

Analysis:
  Weighted Score: 7.56/10
  Bet Direction: YES
  Entry Price: $0.82

Recommendation:
  Action: INVEST
  Risk Grade: 8


---

## 8. Summary: The Complete Pipeline

How the real system works end-to-end.

In [67]:
print("🚀 INSIDER ALPHA PIPELINE — REAL EXECUTION\n")
print("INPUT: 1 real prediction market")
print(f"  Question: {market['question'][:60]}...")
print(f"  Volume: ${market['volume']:,.0f}\n")

print("AGENT A: Insider risk analysis (LLM call)")
print(f"  → Score: {report_a.insider_risk_score}/10")
print(f"  → Confidence: {report_a.confidence}\n")

print("AGENT B: Market behavior analysis (LLM + tools)")
if 'report_b' in locals():
    print(f"  → Score: {report_b.behavior_score}/10")
    print(f"  → Signal: {report_b.signal_direction}")
else:
    print(f"  → (Requires real price history)\n")

print("REVISION AGENT: Pattern detection (LLM call)")
if 'revision_out' in locals():
    print(f"  → Pattern: {revision_out.revision_flag}")
    print(f"  → Recommendation: {revision_out.recommendation_to_decision_agent}\n")

print("DECISION AGENT: Investment decision (deterministic)")
if 'decision' in locals():
    print(f"  → Decision: {'GO' if decision.decision == 'GO' else 'SKIP'}")
    if decision.analysis:
        print(f"  → Edge: {decision.analysis.get('estimated_edge_pp', '?')}pp\n")

print("✅ PIPELINE COMPLETE")
print(f"\n⏱️  Time: ~3 minutes (4 LLM calls × sequential execution)")
print(f"💰 Cost: ~$0.50-1.00 (OpenAI API)")

🚀 INSIDER ALPHA PIPELINE — REAL EXECUTION

INPUT: 1 real prediction market
  Question: Will GPT-4.5 be released by February 28?...
  Volume: $141,504

AGENT A: Insider risk analysis (LLM call)
  → Score: 6/10
  → Confidence: medium

AGENT B: Market behavior analysis (LLM + tools)
  → Score: 9/10
  → Signal: YES
REVISION AGENT: Pattern detection (LLM call)
  → Pattern: NONE
  → Recommendation: GO_EVALUATE

DECISION AGENT: Investment decision (deterministic)
  → Decision: GO
  → Edge: 10.24pp

✅ PIPELINE COMPLETE

⏱️  Time: ~3 minutes (4 LLM calls × sequential execution)
💰 Cost: ~$0.50-1.00 (OpenAI API)


---

## What Just Happened

You ran the **REAL Insider Alpha system** from the polymarket code repository:

1. **Agent A** (real) analyzed insider risk using LLM
2. **Agent B** (real) analyzed market behavior using price tools + LLM
3. **Revision Agent** (real) detected patterns and validated coherence
4. **Decision Agent** (real) weighted both signals and made GO/SKIP decision

No simplifications. No mocked outputs. Real agents, real LLM calls.

**Note:** Agent B normally requires real price history timeseries. This demo shows what happens with limited data — in production with full price data, Agent B would provide detailed behavior scores.